In [11]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [4]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [5]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([8])


In [6]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


In [7]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.shape)

torch.Size([3, 784])


In [8]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


In [9]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.4064,  0.2091, -0.0389, -0.1256,  0.4750, -0.2234, -0.1152,  0.0826,
          0.3719,  0.0272,  0.0399, -0.4668, -0.3570, -0.5464,  0.1105, -0.0926,
         -0.0521, -0.2911,  0.2617, -0.2078],
        [ 0.7153, -0.0258, -0.0221,  0.0774,  0.3155,  0.0232, -0.2643, -0.1152,
          0.4294, -0.0613,  0.0967, -0.3463, -0.1208, -0.1322,  0.2297, -0.1367,
          0.2133, -0.1259,  0.0137, -0.1982],
        [ 0.3396, -0.0678,  0.1832,  0.2395,  0.4407, -0.1593, -0.2029,  0.2402,
          0.5715, -0.1358,  0.1532, -0.5109, -0.2294, -0.1997,  0.1572, -0.1632,
          0.2984, -0.3891,  0.2539, -0.3935]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.4064, 0.2091, 0.0000, 0.0000, 0.4750, 0.0000, 0.0000, 0.0826, 0.3719,
         0.0272, 0.0399, 0.0000, 0.0000, 0.0000, 0.1105, 0.0000, 0.0000, 0.0000,
         0.2617, 0.0000],
        [0.7153, 0.0000, 0.0000, 0.0774, 0.3155, 0.0232, 0.0000, 0.0000, 0.4294,
         0.0000, 0.0967, 0.0000, 0.0000, 0.0000, 0.22

In [15]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

In [16]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

In [17]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0204, -0.0139, -0.0099,  ..., -0.0209,  0.0357,  0.0341],
        [-0.0007,  0.0228, -0.0010,  ..., -0.0107, -0.0109, -0.0022]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([ 0.0037, -0.0060], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0210, -0.0298,  0.0352,  ...,  0.0417, -0.0416, -0.0419],
        [ 0.0325,  0.0057, -0.0384,  ..., -0.0222,  0.0056, -0.0426]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | 